In [1]:
# Ensure we're running from the project root (parent of notebooks/)
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print('Working dir:', Path.cwd())

Working dir: /mnt/d/AI/Projects/slonik-7b


# 03 — GRPO Results & SFT Comparison

After 2000 steps of GRPO with execution-based rewards, Slonik-7B reaches:
- **38.20% on BIRD Mini-Dev PostgreSQL** — beats GPT-4o (34.44%)
- **45.20% on BIRD Mini-Dev SQLite**

This notebook compares SFT and GRPO predictions example-by-example to understand what GRPO actually learned.


## SFT → GRPO trajectory on BIRD-PG

| Stage | Overall | Simple | Moderate | Challenging |
|---|---:|---:|---:|---:|
| Base Qwen2.5-Coder-7B | 12.22% | — | — | — |
| Slonik-7B-SFT | 33.20% | 48.6% | 29.6% | 19.6% |
| **Slonik-7B-GRPO (2000 steps)** | **38.20%** | **56.1%** | **33.6%** | **23.5%** |


## GRPO training configuration

| | |
|---|---|
| Method | GRPO (Group Relative Policy Optimization) |
| Reward — exec_match | weight 1.0 (execution match against BIRD SQLite databases) |
| Reward — syntax_valid | weight 0.2 (sqlglot parse check) |
| Reward — format_correct | weight 0.1 (code-fence wrapping) |
| Steps | 2000 |
| num_generations | 2 |
| LR | 5e-6 |
| Max prompt length | 3072 |
| Max completion length | 512 |
| Execution timeout | 5 sec per query |
| Wall time | 16h (transformers rollouts; vLLM disabled on Blackwell sm_120) |


## Final BIRD-PG and BIRD-SQLite accuracy

In [2]:
import json
from collections import Counter

def load_results(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def stats(results):
    s = Counter(r["status"] for r in results)
    total = s["correct"] + s["wrong"] + s["exec_error"]
    return s, total, s["correct"] / total * 100

grpo_pg = load_results("results/bird_pg_results_grpo2k.jsonl")
grpo_sqlite = load_results("results/bird_sqlite_results_2k.jsonl")

_, t_pg, acc_pg = stats(grpo_pg)
_, t_sqlite, acc_sqlite = stats(grpo_sqlite)

print(f"BIRD-PG     : {acc_pg:.2f}% ({Counter(r['status'] for r in grpo_pg)['correct']}/{t_pg})")
print(f"BIRD-SQLite : {acc_sqlite:.2f}% ({Counter(r['status'] for r in grpo_sqlite)['correct']}/{t_sqlite})")

BIRD-PG     : 38.20% (191/500)
BIRD-SQLite : 45.20% (226/500)


## SFT vs GRPO comparison: who got what right?

The most interesting analysis is example-by-example: which queries did GRPO fix, and which did it break?


In [3]:
sft = {r["question_id"]: r for r in load_results("results/bird_pg_results.jsonl")}
grpo = {r["question_id"]: r for r in load_results("results/bird_pg_results_grpo2k.jsonl")}

fixed = []   # SFT wrong → GRPO right
broken = []  # SFT right → GRPO wrong
same_correct = []
same_wrong = []

for qid in sft:
    if qid not in grpo:
        continue
    s = sft[qid]["status"]
    g = grpo[qid]["status"]
    if s != "correct" and g == "correct":
        fixed.append(qid)
    elif s == "correct" and g != "correct":
        broken.append(qid)
    elif s == "correct" and g == "correct":
        same_correct.append(qid)
    else:
        same_wrong.append(qid)

print(f"Both correct:   {len(same_correct)}")
print(f"GRPO fixed:     {len(fixed)}")
print(f"GRPO broke:     {len(broken)}")
print(f"Both wrong:     {len(same_wrong)}")
print(f"\nNet improvement: +{len(fixed) - len(broken)}")

Both correct:   157
GRPO fixed:     34
GRPO broke:     9
Both wrong:     300

Net improvement: +25


## What GRPO actually fixed — 3 representative wins

### Win 1: Dialect awareness (Q901)

SFT generated `MONTH(date)` — MySQL syntax that fails on Postgres. GRPO learned the PostgreSQL form.


In [4]:
# Show Q901 if it's in our results
q901 = next((r for r in grpo_pg if r["question_id"] == 901), None)
q901_sft = next((r for r in sft.values() if r["question_id"] == 901), None)

if q901 and q901_sft:
    print(f"Question: {q901['question']}")
    print(f"\nGold:\n  {q901['gold_sql']}")
    print(f"\nSFT ({q901_sft['status']}):\n  {q901_sft['pred_sql']}")
    print(f"\nGRPO ({q901['status']}):\n  {q901['pred_sql']}")

Question: Name the races along with its circuit name and location for f1 races hosted in September 2005.

Gold:
  SELECT DISTINCT T2.name, T1.name, T1.location FROM circuits AS T1 INNER JOIN races AS T2 ON T2.circuitID = T1.circuitId WHERE T2.year = 2005 AND TO_CHAR(CAST(T2.date AS TIMESTAMP), 'MM') = '09'

SFT (exec_error):
  SELECT T1.name, T2.name, T2.location FROM races AS T1 JOIN circuits AS T2 ON T1.circuitid = T2.circuitid WHERE MONTH(T1.date) = 9 AND YEAR(T1.date) = 2005

GRPO (correct):
  SELECT T1.name, T2.name, T2.location FROM races AS T1 JOIN circuits AS T2 ON T1.circuitid = T2.circuitid WHERE EXTRACT(MONTH FROM T1.date) = 9 AND EXTRACT(YEAR FROM T1.date) = 2005


### Win 2: Date format reasoning (Q1037)

SFT guessed `LIKE '%/%/87%'` (mm/dd/yy format). The dates were stored YYYY-MM-DD, so SFT got empty results. GRPO settled on `LIKE '%1987%'`.


### Win 3: Identifier quoting (Q1098)

SFT over-quoted `T1."chanceCreationPassingClass"`, breaking case-match against the unquoted DDL. GRPO learned to leave it unquoted.


## What GRPO regressed

6 queries SFT got right, GRPO got wrong. The common pattern: GRPO over-generalized stylistic choices from successful rollouts.


In [5]:
# Display the GRPO regressions
print(f"GRPO broke {len(broken)} queries that SFT had right.\n")
for qid in broken[:6]:
    s = sft[qid]
    g = grpo[qid]
    print(f"=== Q{qid} [{s['difficulty']}] ===")
    print(f"Question: {s['question'][:100]}")
    print(f"SFT (right): {s['pred_sql'][:120]}")
    print(f"GRPO (wrong): {g['pred_sql'][:120]}")
    print()

GRPO broke 9 queries that SFT had right.

=== Q1501 [moderate] ===
Question: Please list the countries of the gas stations with transactions taken place in June, 2013.
SFT (right): SELECT DISTINCT T2.Country FROM yearmonth AS T1 INNER JOIN gasstations AS T2 ON T1.CustomerID = T2.GasStationID WHERE LE
GRPO (wrong): SELECT DISTINCT T1.Country FROM gasstations AS T1 INNER JOIN transactions_1k AS T2 ON T1.GasStationID = T2.GasStationID 

=== Q1155 [simple] ===
Question: List the patient ID, sex and birthday of patient with LDH beyond normal range.
SFT (right): SELECT T1.ID, T1.SEX, T1.Birthday FROM PATIENT AS T1 INNER JOIN Laboratory AS T2 ON T1.ID = T2.ID WHERE T2.LDH > 500
GRPO (wrong): SELECT T1."ID", T1."SEX", T1."Birthday" FROM Laboratory AS T2 INNER JOIN Patient AS T1 ON T1."ID" = T2."ID" WHERE T2."LD

=== Q1208 [moderate] ===
Question: Provide IDs for male patients with ALT glutamic pylvic transaminase (GPT) that have history of ALT g
SFT (right): SELECT T1.ID FROM Patient AS T1 INN

## Takeaway

GRPO with binary execution rewards is powerful for dialect/grounding issues but can't distinguish "succeeded due to better understanding" from "succeeded due to incidental style". Net gain was still +5 points overall and +4 on hardest tier, but the regressions are a real honest limitation worth naming.

A v2 with AST-distance rewards or larger `num_generations` would likely fix several of these.
